# Laboration 1

## Excercise 1

In [1]:
import torch as t

a = t.tensor([2.0, 0.0, 0.0])
b = t.tensor([1.0, 0.25, 0.0])
c = t.tensor([1.0, 1.0, 1.0])

V = a @ (t.linalg.cross(b, c))
print(V)

tensor(0.5000)


## Excercise 2

In [2]:
q = t.tensor([[1, 2], [3, 4], [5, 6]])
a = t.tensor([1, 0, 1])

print(q[t.arange(q.shape[0]), a])

tensor([2, 3, 6])


## Excercise 3

In [3]:
ALPHA = 1e-1
ITERATIONS = 100
xyz = t.tensor([1.0, 1.0, 1.0], dtype = t.float32, requires_grad=True)

for _ in range(ITERATIONS):
    x, y, z = xyz
    f = x**2 - 5*x + y**2 - y + z**2 - 3*z + 8.75
    f.backward()

    with t.no_grad():
        xyz -= ALPHA * xyz.grad
        xyz.grad.zero_()

print(f"Final xyz: {xyz.detach()}")

Final xyz: tensor([2.5000, 0.5000, 1.5000])


In [4]:
import torch.nn as nn


class poly(nn.Module):

    def __init__(self):
        super().__init__()
        self.C = nn.Parameter(t.tensor([1.0, 2.0, 3.0]))
        self.powers = nn.Parameter(t.tensor([0.0, 1.0, 2.0]))

    def forward(self, x):
        x = t.cat([x, x, x], dim=1) ** self.powers
        y = t.einsum("n, n... -> ...", self.C, x.T)
        return y[:, None]


polynomial = poly()

optimizer = t.optim.Adam(polynomial.parameters(), lr=1e-1)

test_x = t.tensor([[0.0987], [0.5131], [1.1002], [1.6118], [1.8777]])
test_y = t.tensor([[1.2265], [2.8162], [6.8318], [12.0178], [15.3323]])

for _ in range(500):
    y_guess = polynomial(test_x)
    loss = t.nn.functional.mse_loss(input=y_guess, target=test_y)
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

print(polynomial.C.detach())

tensor([0.9306, 1.9399, 3.1296])


In [5]:
def greedy_actions(net: nn.Module, states: t.Tensor):
    "Takes batches of states and returns the action for each state."

    action_probabilities = net(states)
    actions = t.argmax(action_probabilities, dim=1)
    return actions


Qnet = nn.Sequential(nn.Linear(10, 20), nn.ReLU(),
                     nn.Linear(20, 40), nn.ReLU(),
                     nn.Linear(40, 20), nn.ReLU(),
                     nn.Linear(20, 6))

# Example

test_states = t.rand([32, 10])
test_actions = greedy_actions(Qnet, test_states)

print(f"Size of input states: {test_states.shape}")
print(f"Size of output actions: {test_actions.shape}")

Size of input states: torch.Size([32, 10])
Size of output actions: torch.Size([32])
